# 04 — FNO Surrogate + BoTorch 다목적 최적화

**목표:** PINN 결과를 빠르게 흉내내는 FNO Surrogate를 학습하고,  
BoTorch qNEHVI로 AR 코팅 두께를 자동 최적화합니다.

| 단계 | 내용 | 소요 시간 |
|------|------|-----------|
| Part 1 | FNO Surrogate 학습 | ~3분 |
| Part 2 | BoTorch 다목적 최적화 | ~5분 |

**3가지 목표:**
- MTF@ridge ↑ (지문 대비도 최대화)
- T_total ↑ (광 투과율 최대화)  
- |skewness| ↓ (PSF 비대칭도 최소화 → 위치 오차 감소)

In [ ]:
# ── 패키지 설치 확인 및 임포트 ────────────────────────────────────────────
import subprocess, sys

def _ensure(pkg, import_as=None):
    name = import_as or pkg
    try:
        __import__(name)
    except ImportError:
        print(f'  pip install {pkg} ...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print('패키지 확인 중...')
for _p, _n in [('torch','torch'), ('numpy','numpy'), ('scipy','scipy'),
                ('matplotlib','matplotlib'), ('botorch','botorch'), ('gpytorch','gpytorch')]:
    _ensure(_p, _n)
    print(f'  ✓ {_p}')

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
from scipy.stats import qmc
import json, time, warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = ['Malgun Gothic', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)
np.random.seed(42)
print(f'\n✓ 준비 완료 | 디바이스: {device}')

## Part 1 — FNO Surrogate
설계 변수 (AR 코팅 층 두께) → PSF 필드를 빠르게 예측하는 신경망을 학습합니다.

In [ ]:
# ── 물리 파라미터 + TMM/ASM/PSF 함수 정의 ────────────────────────────────
wavelength_um = 0.85
k0_um = 2 * np.pi / wavelength_um   # μm⁻¹

# 굴절률 (λ=850nm)
n_SiO2  = 1.47
n_TiO2  = 2.35
n_glass = 1.52
n_air   = 1.00

# Gorilla DX 기준 층 두께 (μm)
BASE_THICK_UM = np.array([49.5, 19.9, 29.6, 130.4]) * 1e-3
LAYER_NS      = [n_SiO2, n_TiO2, n_SiO2, n_TiO2]

# 1D 그리드
N_FIELD   = 512                            # 필드 포인트 수 (FNO 입력 크기)
x_um      = np.linspace(-300.0, 300.0, N_FIELD, dtype=np.float32)  # μm
dx_um     = float(x_um[1] - x_um[0])
sigma_bm  = 100.0   # Gaussian 빔 폭 μm

# ── TMM: 위상 계산 ──
def tmm_phase(d_scales_np, theta_deg):
    """p-편광 TMM → (T_power, phase_rad)"""
    lam_m = wavelength_um * 1e-6
    k0m   = 2 * np.pi / lam_m
    n_in  = n_glass
    n_out = n_air
    th_i  = np.radians(theta_deg)
    d_m   = BASE_THICK_UM * d_scales_np * 1e-6   # μm → m

    # Snell's law 각도
    cos_list = [np.cos(th_i + 0j)]
    for nl in LAYER_NS:
        sn = n_in * np.sin(th_i) / nl
        cos_list.append(np.sqrt(1 - sn**2 + 0j))
    sn_out = n_in * np.sin(th_i) / n_out
    cos_out = np.sqrt(1 - sn_out**2 + 0j)

    # 전달 행렬
    M = np.eye(2, dtype=complex)
    for i, (nl, d) in enumerate(zip(LAYER_NS, d_m)):
        phi = k0m * nl * cos_list[i+1] * d
        m = np.array([
            [np.cos(phi), -1j * np.sin(phi) / (nl * cos_list[i+1])],
            [-1j * nl * cos_list[i+1] * np.sin(phi), np.cos(phi)]
        ], dtype=complex)
        M = M @ m

    p_i = n_in  * cos_list[0]
    p_o = n_out * cos_out
    denom = (M[0,0] + M[0,1]*p_o)*p_i + (M[1,0] + M[1,1]*p_o)
    t = 2 * p_i / (denom + 1e-30)
    T_pow = float(np.real(p_o / p_i * np.abs(t)**2))
    return max(0.0, min(1.0, T_pow)), float(np.angle(t))

# ── ASM: 1D 전파 ──
def asm_1d(U_in, dx, d_um, k0=None, n=1.0):
    k0 = k0 or k0_um
    N  = len(U_in)
    kx = 2 * np.pi * np.fft.fftfreq(N, d=dx)
    kz2 = (n * k0)**2 - kx**2
    kz  = np.sqrt(np.where(kz2 >= 0, kz2, 0) + 0j)
    return np.fft.ifft(np.fft.fft(U_in) * np.exp(1j * kz * d_um))

# ── PSF 지표 계산 ──
def psf_metrics(U_out, x, roi=200.0):
    I = np.abs(U_out)**2
    T = float(np.sum(I) * (x[1]-x[0]))
    # MTF: 중심/사이드로브 비
    c  = I[np.abs(x) < 15.0].mean() if np.any(np.abs(x) < 15.0) else 0
    sl = I[(np.abs(x) > 25.0) & (np.abs(x) < 80.0)].mean() + 1e-10
    MTF = float(c / (c + sl))
    # skewness
    mk  = np.abs(x) <= roi
    xr, Ir = x[mk], I[mk]
    if Ir.sum() > 0:
        Ir /= Ir.sum()
        mu   = (xr * Ir).sum()
        sig2 = ((xr - mu)**2 * Ir).sum()
        skew = float(((xr - mu)**3 * Ir).sum() / (sig2**1.5 + 1e-30))
    else:
        skew = 0.0
    return MTF, T, skew

# ── 설계 변수 → 필드 + 지표 ──
def design_to_field_and_metrics(d_scales, theta_deg):
    """returns (U_in, U_out, MTF, T_total, skewness)"""
    T_c, phi    = tmm_phase(d_scales, theta_deg)
    _, phi0     = tmm_phase(d_scales, 0.0)
    dphi        = phi - phi0
    kx_t        = k0_um * np.sin(np.radians(theta_deg))
    U_in        = (np.exp(-(x_um**2) / (2*sigma_bm**2)) *
                   np.exp(1j * (kx_t * x_um + dphi))).astype(np.complex64)
    U_out       = asm_1d(U_in, dx_um, 30.0).astype(np.complex64)
    MTF, T, sk  = psf_metrics(np.abs(U_out)**2 * T_c, x_um)
    return U_in, U_out, MTF, T * T_c, sk

# 동작 확인
_, _, MTF0, T0, sk0 = design_to_field_and_metrics(np.ones(4), 30.0)
print('✓ 물리 함수 정의 완료')
print(f'  기준 DX (θ=30°): MTF={MTF0:.3f}  T={T0:.4f}  skew={sk0:.4f}')

In [ ]:
# ── FNO 아키텍처 정의 ────────────────────────────────────────────────────
# 입력 채널 (7채널):
#   0: Re(U_in)     1: Im(U_in)
#   2~5: d1,d2,d3,d4 (각 층 두께 배율, broadcast)
#   6: theta/30 (정규화된 입사각, broadcast)
# 출력 채널 (2채널): Re(U_out), Im(U_out)

import torch.nn.functional as F   # gelu 등 함수형 API

FNO_IN_CH  = 7    # 입력 채널 수
FNO_OUT_CH = 2    # 출력 채널 수
FNO_WIDTH  = 32   # 숨겨진 채널 수
FNO_MODES  = 64   # 유지할 푸리에 모드 수
FNO_BLOCKS = 4    # FNO 블록 수


class SpectralConv1d(nn.Module):
    """푸리에 공간에서 학습 가능한 필터를 적용하는 레이어"""
    def __init__(self, in_ch, out_ch, modes):
        super().__init__()
        self.modes = modes
        scale = 1.0 / (in_ch * out_ch)
        self.W_r = nn.Parameter(scale * torch.randn(in_ch, out_ch, modes))
        self.W_i = nn.Parameter(scale * torch.randn(in_ch, out_ch, modes))

    def forward(self, x):
        B, _, N = x.shape
        x_ft  = torch.fft.rfft(x, dim=-1)
        modes = min(self.modes, x_ft.shape[-1])

        xr = x_ft[:, :, :modes].real
        xi = x_ft[:, :, :modes].imag
        out_r = (torch.einsum('bim,iom->bom', xr, self.W_r)
                 - torch.einsum('bim,iom->bom', xi, self.W_i))
        out_i = (torch.einsum('bim,iom->bom', xr, self.W_i)
                 + torch.einsum('bim,iom->bom', xi, self.W_r))

        out_ft = torch.zeros(B, self.W_r.shape[1], x_ft.shape[-1],
                             dtype=torch.cfloat, device=x.device)
        out_ft[:, :, :modes] = torch.complex(out_r, out_i)
        return torch.fft.irfft(out_ft, n=N, dim=-1)


class FNOBlock(nn.Module):
    """FNO 블록: 스펙트럴 레이어 + bypass + 정규화 + 활성화"""
    def __init__(self, width, modes):
        super().__init__()
        self.spectral = SpectralConv1d(width, width, modes)
        self.bypass   = nn.Conv1d(width, width, kernel_size=1)
        self.norm     = nn.InstanceNorm1d(width)

    def forward(self, x):
        return F.gelu(self.norm(self.spectral(x) + self.bypass(x)))


class FNO1d(nn.Module):
    """1D Fourier Neural Operator"""
    def __init__(self, in_ch, out_ch, width, modes, n_blocks):
        super().__init__()
        self.lift   = nn.Conv1d(in_ch, width, kernel_size=1)
        self.blocks = nn.ModuleList([FNOBlock(width, modes) for _ in range(n_blocks)])
        self.proj1  = nn.Conv1d(width, 64, kernel_size=1)
        self.proj2  = nn.Conv1d(64, out_ch, kernel_size=1)

    def forward(self, x):
        x = self.lift(x)
        for blk in self.blocks:
            x = blk(x)
        return self.proj2(F.gelu(self.proj1(x)))


# 모델 생성
fno_model = FNO1d(
    in_ch=FNO_IN_CH, out_ch=FNO_OUT_CH,
    width=FNO_WIDTH, modes=FNO_MODES, n_blocks=FNO_BLOCKS
).to(device)

n_params = sum(p.numel() for p in fno_model.parameters())
print('✓ FNO 모델 생성 완료')
print(f'  입력 채널: {FNO_IN_CH}  (Re/Im(U_in) + d1~d4 + theta)')
print(f'  출력 채널: {FNO_OUT_CH}  (Re/Im(U_out))')
print(f'  파라미터 수: {n_params:,}')

# 동작 확인
_dummy = torch.randn(2, FNO_IN_CH, N_FIELD).to(device)
_out   = fno_model(_dummy)
assert _out.shape == (2, FNO_OUT_CH, N_FIELD), f'shape 오류: {_out.shape}'
print(f'  Forward 확인: (2,{FNO_IN_CH},{N_FIELD}) → {tuple(_out.shape)}  ✓')
del _dummy, _out


In [ ]:
# ── 학습 데이터 생성 (LHS 샘플링) ────────────────────────────────────────
N_TRAIN   = 300
N_VAL     = 60
N_TOTAL   = N_TRAIN + N_VAL
THETA_SET = [0, 5, 10, 15, 20, 25, 30]

print(f'학습 데이터 생성 중... (총 {N_TOTAL}개)')
t0 = time.time()

# Latin Hypercube Sampling: 4차원 설계 변수 [0.7, 1.3] 범위
sampler  = qmc.LatinHypercube(d=4, seed=0)
lhs_unit = sampler.random(N_TOTAL)
D_all    = 0.7 + lhs_unit * 0.6    # (N, 4), 범위 [0.7, 1.3]
theta_all = np.array([THETA_SET[i % len(THETA_SET)] for i in range(N_TOTAL)],
                     dtype=np.float32)
np.random.shuffle(theta_all)

# 필드 + 지표 계산
X_list, Y_list, M_list = [], [], []

for i in range(N_TOTAL):
    d_sc  = D_all[i]
    theta = float(theta_all[i])
    U_in, U_out, MTF, T, sk = design_to_field_and_metrics(d_sc, theta)

    # FNO 입력 채널 구성 (N_FIELD × 9)
    ones = np.ones(N_FIELD, dtype=np.float32)
    ch_Ur   = U_in.real.astype(np.float32)
    ch_Ui   = U_in.imag.astype(np.float32)
    ch_d    = [ones * float(d_sc[j]) for j in range(4)]   # 4층 두께 broadcast
    ch_th   = ones * (theta / 30.0)                        # theta 정규화
    x_in    = np.stack([ch_Ur, ch_Ui] + ch_d + [ch_th], axis=0)  # (9, N)

    # FNO 출력 채널 (N_FIELD × 2)
    y_out   = np.stack([U_out.real.astype(np.float32),
                        U_out.imag.astype(np.float32)], axis=0)   # (2, N)

    X_list.append(x_in)
    Y_list.append(y_out)
    M_list.append([MTF, T, sk])   # PSF 지표 (BoTorch용)

    if (i+1) % 100 == 0:
        print(f'  [{i+1}/{N_TOTAL}] ({time.time()-t0:.1f}s)')

X_arr = np.array(X_list, dtype=np.float32)   # (N, 9, N_FIELD)
Y_arr = np.array(Y_list, dtype=np.float32)   # (N, 2, N_FIELD)
M_arr = np.array(M_list, dtype=np.float32)   # (N, 3)

# train / val 분리
X_tr, X_vl = X_arr[:N_TRAIN], X_arr[N_TRAIN:]
Y_tr, Y_vl = Y_arr[:N_TRAIN], Y_arr[N_TRAIN:]

print(f'\n✓ 데이터 생성 완료 ({time.time()-t0:.1f}s)')
print(f'  훈련: {X_tr.shape}  검증: {X_vl.shape}')
print(f'  MTF  범위: [{M_arr[:,0].min():.3f}, {M_arr[:,0].max():.3f}]')
print(f'  T    범위: [{M_arr[:,1].min():.4f}, {M_arr[:,1].max():.4f}]')
print(f'  skew 범위: [{M_arr[:,2].min():.3f}, {M_arr[:,2].max():.3f}]')

In [ ]:
# ── FNO 훈련 ─────────────────────────────────────────────────────────────
BATCH_SIZE = 32
N_EPOCHS   = 200
LR         = 3e-4

# DataLoader
Xt = torch.from_numpy(X_tr).to(device)
Yt = torch.from_numpy(Y_tr).to(device)
Xv = torch.from_numpy(X_vl).to(device)
Yv = torch.from_numpy(Y_vl).to(device)

train_loader = DataLoader(TensorDataset(Xt, Yt), batch_size=BATCH_SIZE, shuffle=True)

optimizer  = optim.AdamW(fno_model.parameters(), lr=LR, weight_decay=1e-4)
scheduler  = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)
loss_fn    = nn.MSELoss()

train_hist, val_hist = [], []
print(f'FNO 훈련 시작 ({N_EPOCHS} epochs)...')
t0 = time.time()

for epoch in range(N_EPOCHS):
    fno_model.train()
    ep_loss = 0.0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        pred = fno_model(xb)
        loss = loss_fn(pred, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(fno_model.parameters(), 1.0)
        optimizer.step()
        ep_loss += loss.item()
    scheduler.step()

    fno_model.eval()
    with torch.no_grad():
        val_loss = loss_fn(fno_model(Xv), Yv).item()

    train_hist.append(ep_loss / len(train_loader))
    val_hist.append(val_loss)

    if (epoch+1) % 50 == 0:
        print(f'  [{epoch+1:3d}/{N_EPOCHS}]  train={train_hist[-1]:.5f}  '
              f'val={val_hist[-1]:.5f}  ({time.time()-t0:.0f}s)')

# 손실 곡선
fig, ax = plt.subplots(figsize=(8, 3))
ax.semilogy(train_hist, 'b-', lw=1.5, label='Train')
ax.semilogy(val_hist,   'r-', lw=1.5, label='Val')
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss')
ax.set_title('FNO 훈련 손실'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('FNO_훈련손실.png', dpi=120)
plt.show()
print(f'\n✓ FNO 훈련 완료 ({time.time()-t0:.1f}s)')
print(f'  최종 Val MSE: {val_hist[-1]:.6f}')

In [ ]:
# ── FNO 검증 + 속도 비교 ─────────────────────────────────────────────────
fno_model.eval()

idx = 5
with torch.no_grad():
    pred_field = fno_model(Xv[idx:idx+1]).cpu().numpy()[0]   # (2, N) — tensor → numpy

true_field = Y_vl[idx]   # 이미 numpy array, .cpu() 불필요

I_pred = pred_field[0]**2 + pred_field[1]**2
I_true = true_field[0]**2 + true_field[1]**2

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
axes[0].plot(x_um, I_true / I_true.max(), 'b-', lw=2, label='ASM (정답)')
axes[0].plot(x_um, I_pred / I_pred.max(), 'r--', lw=1.5, label='FNO (예측)')
axes[0].set_xlim(-200, 200); axes[0].set_xlabel('x (μm)')
axes[0].set_title('PSF 비교'); axes[0].legend(); axes[0].grid(alpha=0.3)

err = np.abs(I_pred - I_true)
axes[1].plot(x_um, err, 'k-', lw=1.2)
axes[1].set_xlim(-200, 200); axes[1].set_xlabel('x (μm)')
axes[1].set_title(f'절대 오차 (max={err.max():.4f})')
axes[1].grid(alpha=0.3)

# 속도 측정 — FNO
N_REPEAT = 500
dummy_in = torch.randn(1, FNO_IN_CH, N_FIELD, device=device)
with torch.no_grad():
    t_start = time.perf_counter()
    for _ in range(N_REPEAT):
        fno_model(dummy_in)
    t_fno = (time.perf_counter() - t_start) / N_REPEAT * 1000   # ms

# 속도 측정 — ASM
U_dummy = np.ones(N_FIELD, dtype=np.complex64)
t_start = time.perf_counter()
for _ in range(N_REPEAT):
    asm_1d(U_dummy, dx_um, 30.0)
t_asm = (time.perf_counter() - t_start) / N_REPEAT * 1000

axes[2].bar(['ASM\n(기존)', 'FNO\n(서로게이트)'], [t_asm, t_fno],
            color=['steelblue', 'tomato'], alpha=0.8)
axes[2].set_ylabel('추론 시간 (ms)'); axes[2].set_title('추론 속도 비교')
for i, v in enumerate([t_asm, t_fno]):
    axes[2].text(i, v * 1.02, f'{v:.3f}ms', ha='center', fontsize=10)
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('FNO_검증결과.png', dpi=120)
plt.show()

print(f'✓ FNO 검증 완료')
print(f'  ASM 추론: {t_asm:.3f} ms')
print(f'  FNO 추론: {t_fno:.3f} ms  (목표: <1ms)')


## Part 2 — BoTorch qNEHVI 다목적 최적화

FNO Surrogate를 사용해 AR 코팅 층 두께를 자동으로 최적화합니다.

- **설계 변수:** 4층 두께 배율 (d1~d4, 범위 [0.7, 1.3])
- **목표 1:** MTF@ridge ↑  
- **목표 2:** T_total ↑  
- **목표 3:** −|skewness| ↑ (부호 반전 = 최소화)

**qNEHVI** (q-Noisy Expected Hypervolume Improvement): 여러 목표를 동시에 최적화하는 베이지안 방법

In [ ]:
# ── BoTorch 설정 ──────────────────────────────────────────────────────────
import botorch
from botorch.models import SingleTaskGP
from botorch.models.transforms.outcome import Standardize
from botorch.fit import fit_gpytorch_mll
from botorch.utils.multi_objective.pareto import is_non_dominated
from botorch.utils.multi_objective.hypervolume import Hypervolume
from botorch.optim import optimize_acqf
import gpytorch

# 다목적 최적화용 FNO 평가 함수
def fno_evaluate(d_scales_np, theta_deg=30.0):
    """(4,) 설계 변수 → [MTF, T, -|skew|] (모두 높을수록 좋음)"""
    U_in, U_out, MTF, T, sk = design_to_field_and_metrics(d_scales_np, theta_deg)
    return np.array([MTF, T, -abs(sk)], dtype=np.float32)

# 초기 데이터: 기존 생성 데이터에서 theta=30°에 해당하는 샘플 추출
theta_mask = (theta_all[:N_TRAIN] == 30.0)
D_init = D_all[:N_TRAIN][theta_mask]
M_init_raw = M_arr[:N_TRAIN][theta_mask]   # [MTF, T, skew]

# 목표값 변환: [MTF, T, -|skew|] (모두 최대화)
Y_init = np.column_stack([
    M_init_raw[:, 0],          # MTF
    M_init_raw[:, 1],          # T
    -np.abs(M_init_raw[:, 2])  # -|skewness|
]).astype(np.float32)

# 초기 데이터가 너무 적으면 보충
if len(D_init) < 15:
    extra_sampler = qmc.LatinHypercube(d=4, seed=99)
    extra_d = 0.7 + extra_sampler.random(20) * 0.6
    extra_y = np.array([fno_evaluate(d) for d in extra_d])
    D_init  = np.vstack([D_init, extra_d])
    Y_init  = np.vstack([Y_init, extra_y])

# Torch 텐서로 변환 (BoTorch는 double 선호)
tkwargs = {'dtype': torch.double, 'device': torch.device('cpu')}
train_X = torch.tensor(D_init, **tkwargs)
train_Y = torch.tensor(Y_init, **tkwargs)

# 입력 범위 정의
bounds = torch.tensor([[0.7]*4, [1.3]*4], **tkwargs)   # (2, 4)

# Reference point (목표: 이 값보다 모두 좋아야 파레토)
ref_point = train_Y.min(dim=0).values - 0.05

print('✓ BoTorch 설정 완료')
print(f'  초기 데이터: {train_X.shape[0]}개 샘플')
print(f'  설계 변수: 4개 (d1~d4 배율)')
print(f'  최적화 목표: MTF↑  T↑  -|skewness|↑')
print(f'  Reference point: {ref_point.numpy().round(4)}')

In [ ]:
# ── qNEHVI 최적화 루프 ───────────────────────────────────────────────────
N_BO_ITER   = 15   # BO 반복 횟수
BATCH_Q     = 3    # 한 번에 제안할 설계 수
N_RESTARTS  = 5
RAW_SAMPLES = 128

hv_history = []
print(f'qNEHVI 다목적 최적화 시작 ({N_BO_ITER} iter × q={BATCH_Q})...')
t0 = time.time()

for bo_iter in range(N_BO_ITER):

    # ── GP 모델 학습 ──
    model = SingleTaskGP(
        train_X, train_Y,
        outcome_transform=Standardize(m=train_Y.shape[-1])
    )
    mll = gpytorch.mlls.ExactMarginalLogLikelihood(model.likelihood, model)
    fit_gpytorch_mll(mll)
    model.eval()

    # ── qNEHVI 획득 함수 ──
    try:
        from botorch.acquisition.multi_objective.logei import (
            qLogNoisyExpectedHypervolumeImprovement as qNEHVI
        )
    except ImportError:
        from botorch.acquisition.multi_objective import (
            qNoisyExpectedHypervolumeImprovement as qNEHVI
        )

    acqf = qNEHVI(
        model       = model,
        ref_point   = ref_point,
        X_baseline  = train_X,
        prune_baseline = True,
        cache_root  = False,
    )

    # ── 다음 후보 탐색 ──
    candidates, _ = optimize_acqf(
        acq_function = acqf,
        bounds       = bounds,
        q            = BATCH_Q,
        num_restarts = N_RESTARTS,
        raw_samples  = RAW_SAMPLES,
    )

    # ── 후보 평가 (FNO Surrogate 사용) ──
    new_X = candidates.detach()
    new_Y_list = []
    for cand in new_X.numpy():
        y = fno_evaluate(cand, theta_deg=30.0)
        new_Y_list.append(y)
    new_Y = torch.tensor(np.array(new_Y_list), **tkwargs)

    # ── 데이터 업데이트 ──
    train_X = torch.cat([train_X, new_X], dim=0)
    train_Y = torch.cat([train_Y, new_Y], dim=0)

    # ── Hypervolume 계산 ──
    pareto_mask = is_non_dominated(train_Y)
    hv_calc = Hypervolume(ref_point=ref_point)
    hv_val  = hv_calc.compute(train_Y[pareto_mask])
    hv_history.append(hv_val)

    print(f'  [iter {bo_iter+1:2d}/{N_BO_ITER}]  '
          f'HV={hv_val:.4f}  '
          f'파레토 점: {pareto_mask.sum().item()}개  '
          f'({time.time()-t0:.0f}s)')

print(f'\n✓ 최적화 완료 ({time.time()-t0:.1f}s)')
print(f'  최종 HV: {hv_history[-1]:.5f}')
print(f'  총 평가 수: {train_X.shape[0]}개')

In [ ]:
# ── 파레토 프론트 시각화 ──────────────────────────────────────────────────
pareto_mask = is_non_dominated(train_Y)
Y_pareto    = train_Y[pareto_mask].numpy()
D_pareto    = train_X[pareto_mask].numpy()

fig = plt.figure(figsize=(15, 4))

# 서브플롯 1: MTF vs T
ax1 = fig.add_subplot(131)
ax1.scatter(train_Y[:,1].numpy(), train_Y[:,0].numpy(),
            c='lightgray', s=20, alpha=0.5, label='탐색 점')
ax1.scatter(Y_pareto[:,1], Y_pareto[:,0],
            c='red', s=60, zorder=5, label='파레토 최적')
ax1.set_xlabel('T_total (투과율)'); ax1.set_ylabel('MTF@ridge')
ax1.set_title('MTF vs 투과율'); ax1.legend(); ax1.grid(alpha=0.3)

# 서브플롯 2: MTF vs |skewness|
ax2 = fig.add_subplot(132)
ax2.scatter(-train_Y[:,2].numpy(), train_Y[:,0].numpy(),
            c='lightgray', s=20, alpha=0.5)
ax2.scatter(-Y_pareto[:,2], Y_pareto[:,0], c='red', s=60, zorder=5)
ax2.set_xlabel('|skewness|'); ax2.set_ylabel('MTF@ridge')
ax2.set_title('MTF vs PSF 비대칭도'); ax2.grid(alpha=0.3)

# 서브플롯 3: Hypervolume 수렴 곡선
ax3 = fig.add_subplot(133)
ax3.plot(range(1, len(hv_history)+1), hv_history, 'b-o', lw=2, ms=6)
ax3.set_xlabel('BO 반복 횟수'); ax3.set_ylabel('Hypervolume')
ax3.set_title('Hypervolume 수렴'); ax3.grid(alpha=0.3)

plt.suptitle('다목적 최적화 결과 (θ=30°, Gorilla DX)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('파레토_프론트.png', dpi=120, bbox_inches='tight')
plt.show()

# 파레토 최적 설계 출력
print(f'\n파레토 최적 설계 ({len(Y_pareto)}개):')
print(f'{"#":>3}  {"MTF":>7}  {"T":>8}  {"skew":>8}  {"d_scales":>30}')
print('-'*65)
sort_idx = np.argsort(-Y_pareto[:,0])   # MTF 내림차순
for rank, i in enumerate(sort_idx[:10]):
    d  = D_pareto[i]
    y  = Y_pareto[i]
    sk = -y[2]
    print(f'{rank+1:3d}  {y[0]:7.4f}  {y[1]:8.5f}  {sk:8.4f}  '
          f'[{d[0]:.3f},{d[1]:.3f},{d[2]:.3f},{d[3]:.3f}]')

In [ ]:
# ── 최적 설계 저장 ────────────────────────────────────────────────────────
# MTF 기준 최상위 파레토 설계 선택
best_idx  = np.argmax(Y_pareto[:, 0])   # MTF 최대
best_d    = D_pareto[best_idx]
best_y    = Y_pareto[best_idx]
best_skew = float(-best_y[2])

# 최적 두께 복원 (μm)
best_thick_um = BASE_THICK_UM * best_d
best_thick_nm = best_thick_um * 1e3

# 결과 딕셔너리
result = {
    'optimization': 'qNEHVI',
    'theta_deg': 30.0,
    'coating': 'Gorilla DX (4-layer)',
    'best_design': {
        'd_scales'    : best_d.tolist(),
        'thickness_nm': best_thick_nm.tolist(),
        'layers'      : [
            {'material': 'SiO2', 'thickness_nm': best_thick_nm[0]},
            {'material': 'TiO2', 'thickness_nm': best_thick_nm[1]},
            {'material': 'SiO2', 'thickness_nm': best_thick_nm[2]},
            {'material': 'TiO2', 'thickness_nm': best_thick_nm[3]},
        ]
    },
    'performance': {
        'MTF_ridge'   : float(best_y[0]),
        'T_total'     : float(best_y[1]),
        'skewness'    : best_skew,
    },
    'baseline_performance': {
        'MTF_ridge'   : float(MTF0),
        'T_total'     : float(T0),
        'skewness'    : float(sk0),
    },
    'pareto_count': int(pareto_mask.sum()),
    'hv_final'    : float(hv_history[-1]),
}

with open('optimal_design.json', 'w', encoding='utf-8') as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

# 모델 저장
torch.save(fno_model.state_dict(), 'fno_model.pt')

print('\n' + '='*55)
print('  최적화 결과 요약')
print('='*55)
print(f'  [기준 DX]  MTF={MTF0:.4f}  T={T0:.5f}  skew={sk0:.4f}')
print(f'  [최적 설계] MTF={best_y[0]:.4f}  T={best_y[1]:.5f}  skew={best_skew:.4f}')
print(f'  MTF  개선: {(best_y[0]-MTF0)/MTF0*100:+.1f}%')
print(f'  skew 개선: {(abs(best_skew)-abs(sk0))/abs(sk0)*100:+.1f}%')
print(f'  최적 두께 (nm): {best_thick_nm.round(1).tolist()}')
print('='*55)
print('  저장: optimal_design.json, fno_model.pt')
print('\n  Phase 1 완료! 다음: L6 인프라 (A100 GPU) 설정')

## Part 3 — BM delta_BM 스윕으로 skewness 최소화\n\nPSF 비대칭(skewness)은 코팅이 아닌 **BM 구조**에서 발생합니다.  \nBM1을 x 방향으로 살짝 이동(delta_BM)하면 θ=30° 기울어진 빛을 보정할 수 있습니다.\n\n```\nθ=30° 빛 →  /  BM2(z=0~10μm)    BM1(z=30~40μm)\n            /      [  gap  ]        [  gap+δ  ]\n```"

In [ ]:
# ── BM delta_BM 스윕 — skewness 최소화 ──────────────────────────────────
# 최적 코팅 파라미터 로드
with open('optimal_design.json', 'r') as _f:
    _opt_data = json.load(_f)
d_scales_opt = np.array(_opt_data['best_design']['d_scales'])

# BM 파라미터 (03_PINN_Helmholtz 와 동일)
w1_bm  = 30.0   # BM1 아퍼처 폭 (μm)
w2_bm  = 40.0   # BM2 아퍼처 폭 (μm)

def compute_psf_with_delta(delta_bm, d_scales, theta_deg=30.0):
    """
    BM 오프셋 포함 1D 광전파 모델:
    U_in × BM2 마스크 → ASM 전파 → BM1 마스크 → PSF
    """
    T_c, phi = tmm_phase(d_scales, theta_deg)
    _, phi0  = tmm_phase(d_scales, 0.0)
    dphi     = phi - phi0
    kx_t     = k0_um * np.sin(np.radians(theta_deg))

    U_in = (np.exp(-x_um**2 / (2*sigma_bm**2)) *
            np.exp(1j * (kx_t * x_um + dphi))).astype(np.complex64)

    # BM2 마스크: |x| > w2/2 차단 (입력면)
    U_in = U_in * (np.abs(x_um) <= w2_bm / 2)

    # ASM 전파 (z=0 → z=30μm)
    U_out = asm_1d(U_in, dx_um, 30.0)

    # BM1 마스크: |x − delta_bm| > w1/2 차단 (출력면)
    U_out = U_out * (np.abs(x_um - delta_bm) <= w1_bm / 2)

    # PSF 강도
    I = np.abs(U_out)**2 * T_c

    # skewness 계산
    roi  = np.abs(x_um) <= 200.0
    xr, Ir = x_um[roi], I[roi]
    if Ir.sum() > 1e-20:
        In   = Ir / Ir.sum()
        mu   = (xr * In).sum()
        sig2 = ((xr - mu)**2 * In).sum()
        skew = float(((xr - mu)**3 * In).sum() / (sig2**1.5 + 1e-30))
    else:
        skew = 0.0
    return I, skew

# ── 스윕: delta_BM = -20 ~ +20 μm ──
delta_range = np.linspace(-20.0, 20.0, 81)
print('delta_BM 스윕 중...')
skew_arr = np.array([compute_psf_with_delta(d, d_scales_opt)[1]
                     for d in delta_range])

# 최적 delta_BM (|skewness| 최소)
best_idx   = int(np.argmin(np.abs(skew_arr)))
best_delta = float(delta_range[best_idx])
best_skew  = float(skew_arr[best_idx])

# ── 시각화 ──
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.plot(delta_range, skew_arr, 'b-o', lw=2, ms=3, label='skewness')
ax.axhline(0,           color='k',  ls='--', lw=1)
ax.axhline( 0.10,       color='orange', ls=':', lw=1.2, label='목표 +0.10')
ax.axhline(-0.10,       color='orange', ls=':', lw=1.2, label='목표 -0.10')
ax.axvline(best_delta,  color='r',  ls='--', lw=2,
           label=f'최적 δ={best_delta:.1f}μm  (skew={best_skew:.3f})')
ax.set_xlabel('delta_BM (μm)')
ax.set_ylabel('PSF skewness')
ax.set_title('BM1 오프셋 vs PSF skewness')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# PSF 비교 (delta=0 vs 최적)
I_base, sk_base = compute_psf_with_delta(0.0,         d_scales_opt)
I_opt,  sk_opt  = compute_psf_with_delta(best_delta,  d_scales_opt)

ax2 = axes[1]
ax2.plot(x_um, I_base / I_base.max(), 'b-',  lw=2,
         label=f'δ=0 μm  (skew={sk_base:.3f})')
ax2.plot(x_um, I_opt  / I_opt.max(),  'r-',  lw=2,
         label=f'δ={best_delta:.1f}μm (skew={sk_opt:.4f})')
ax2.set_xlim(-200, 200)
ax2.set_xlabel('x (μm)'); ax2.set_ylabel('정규화 강도')
ax2.set_title('PSF 비교 (BM 오프셋 전/후)')
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('BM_delta_최적화.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'\n✓ BM delta 최적화 완료')
print(f'  최적 delta_BM : {best_delta:.1f} μm')
print(f'  δ=0    skewness: {sk_base:.4f}')
print(f'  δ최적  skewness: {sk_opt:.4f}')
print(f'  목표: |skewness| < 0.10')